### Read experimental result data

In [2]:
# read parquet file
import polars as pl
# df = pl.read_parquet("../../experiments/16_10_result.parquet")
df = pl.read_parquet("/Users/shinomilab-mac-pro/seokyeong/diffusion_seokyeong/test/twitch_ES.parquet")
print(df)

# read metadata of the parquet file
import pyarrow.parquet as pq
# meta = pq.read_metadata("../test/result.parquet")
meta = pq.read_metadata("/Users/shinomilab-mac-pro/seokyeong/diffusion_seokyeong/test/twitch_ES.parquet")

# You can check versions depending on libraries 
print(meta.metadata)

shape: (524_288, 5)
┌─────┬──────┬───────┬───────┬───────────────┐
│ id  ┆ ip   ┆ us_id ┆ msg   ┆ mean_num_xact │
│ --- ┆ ---  ┆ ---   ┆ ---   ┆ ---           │
│ u64 ┆ u64  ┆ u64   ┆ str   ┆ f64           │
╞═════╪══════╪═══════╪═══════╪═══════════════╡
│ 0   ┆ 1349 ┆ 0     ┆ 30000 ┆ 0.0           │
│ 0   ┆ 1349 ┆ 0     ┆ 31000 ┆ 0.06          │
│ 0   ┆ 1349 ┆ 0     ┆ 11000 ┆ 0.0           │
│ 0   ┆ 1349 ┆ 0     ┆ 21000 ┆ 0.02          │
│ 0   ┆ 1349 ┆ 0     ┆ 01000 ┆ 0.0           │
│ …   ┆ …    ┆ …     ┆ …     ┆ …             │
│ 511 ┆ 2821 ┆ 31    ┆ 22333 ┆ 1.34          │
│ 511 ┆ 2821 ┆ 31    ┆ 13333 ┆ 1.44          │
│ 511 ┆ 2821 ┆ 31    ┆ 03333 ┆ 1.31          │
│ 511 ┆ 2821 ┆ 31    ┆ 23333 ┆ 1.52          │
│ 511 ┆ 2821 ┆ 31    ┆ 33333 ┆ 1.32          │
└─────┴──────┴───────┴───────┴───────────────┘
{b'version_runner': b'0.1.1', b'version_core': b'0.1.1', b'version': b'0.1.0', b'ARROW:schema': b'/////0wBAAAQAAAAAAAKAAwACgAJAAQACgAAABAAAAAAAQQACAAIAAAABAAIAAAABAAAAAUAAADsAAAAsAA

### Compute solutions and/or values of optimal, RAM, and URS policies

In [3]:
EV_BY_OPT = "ev_by_opt"
EV_BY_RAM = "ev_by_ram"
EV_BY_URS = "ev_by_urs"
OPT_MEAN = "mean_of_opt_values"
RAMEV_OPTEV_RATIO = "ram ev / opt ev"
URSEV_OPTEV_RATIO = "urs ev / opt ev"
RAMEV_OPTM_RATIO = "ram ev / opt mean"
URSEV_OPTM_RATIO = "urs ev / opt mean"

MEAN_NUM_XACT = "mean_num_xact"

lf_opt = (
    df.lazy()
    .filter(pl.col(MEAN_NUM_XACT) == pl.col(MEAN_NUM_XACT).max().over(["us_id", "ip"]))
    .unique(["ip", "us_id", MEAN_NUM_XACT])
    .select([pl.col("us_id"), pl.col("ip"), pl.col("msg").name.prefix("argmax_"), pl.col(MEAN_NUM_XACT).name.prefix("max_")])
)
lf_opt_mod = (df.lazy()
    .join(lf_opt, left_on=pl.col("msg"), right_on=pl.col("argmax_msg"))
    .group_by("ip_right", "us_id_right")
    .agg(pl.col(MEAN_NUM_XACT).mean())
    .join(lf_opt, left_on=["ip_right", "us_id_right"], right_on=["ip", "us_id"])
    .select([MEAN_NUM_XACT, "max_" + MEAN_NUM_XACT])
    .mean()
    .select([pl.col(MEAN_NUM_XACT).alias(EV_BY_OPT), pl.col("max_" + MEAN_NUM_XACT).alias(OPT_MEAN)])
)
lf_rob = (
    df.lazy()
    .group_by("msg")
    .agg(pl.col(MEAN_NUM_XACT).mean())
    .filter(pl.col(MEAN_NUM_XACT) == pl.col(MEAN_NUM_XACT).max())
    .select(pl.col(MEAN_NUM_XACT).alias(EV_BY_RAM))
)
lf_urs = (
    df.lazy()
    .select(MEAN_NUM_XACT)
    .mean()
    .select(pl.col(MEAN_NUM_XACT).alias(EV_BY_URS))
)
eval_df = (
    pl.concat([lf_opt_mod, lf_rob, lf_urs], how="horizontal")
    .with_columns([
        (pl.col(EV_BY_RAM) / pl.col(EV_BY_OPT)).alias(RAMEV_OPTEV_RATIO),
        (pl.col(EV_BY_URS) / pl.col(EV_BY_OPT)).alias(URSEV_OPTEV_RATIO),
        (pl.col(EV_BY_RAM) / pl.col(OPT_MEAN)).alias(RAMEV_OPTM_RATIO),
        (pl.col(EV_BY_URS) / pl.col(OPT_MEAN)).alias(URSEV_OPTM_RATIO),
    ])
).collect().transpose(include_header=True).rename({"column": "key", "column_0": "value"})

In [4]:
lf_opt_mod = (df.lazy()
    .join(lf_opt, left_on=pl.col("msg"), right_on=pl.col("argmax_msg"))#普通のdfにlf_optで事後確認した最適なmsgを他にip,us_idに適用されたのがあればそれをあわせて平均化。
    .group_by("ip_right", "us_id_right")#他のip,user_stateにもoptのmsgが通用するかという問い
    .agg(pl.col(MEAN_NUM_XACT).mean())
    .join(lf_opt, left_on=["ip_right", "us_id_right"], right_on=["ip", "us_id"])#ipとus_idにピッタリなmsg
    .mean()
    .select([pl.col(MEAN_NUM_XACT).alias("ip,user_state関係なく、最適なmsgをランダムに適用したxact期待値"),
             pl.col("max_" + MEAN_NUM_XACT).alias("ipとuser_stateに最適なmsg適用後の期待値")])
)
lf_opt_mod.mean().collect()

"ip,user_state関係なく、最適なmsgをランダムに適用したxact期待値",ipとuser_stateに最適なmsg適用後の期待値
f64,f64
36.35044,55.363691


ipだけぴったりのmsgだけ適用すると?

In [5]:
ip_opt = (
    df.lazy()
    .filter(pl.col(MEAN_NUM_XACT) == pl.col(MEAN_NUM_XACT).max().over(["ip"]))
    .unique(["ip", MEAN_NUM_XACT])
    .select([pl.col("ip"), pl.col("msg"), pl.col(MEAN_NUM_XACT)])
).collect()


In [6]:
ip_opt

ip,msg,mean_num_xact
u64,str,f64
775,"""00023""",827.19
153,"""10033""",32.83
4357,"""20133""",8.51
2001,"""00103""",19.84
1349,"""20201""",0.93
…,…,…
1035,"""33222""",1.0
2823,"""30033""",201.71
2606,"""00023""",279.49


In [7]:
ip_opt.select(pl.col("mean_num_xact").mean().alias("ip_opt_msg_xact"))#平均値がipとuser_stateより高い

ip_opt_msg_xact
f64
182.250625


In [8]:
ip_opt.group_by("msg").len()

msg,len
str,u32
"""30033""",1
"""00013""",2
"""01113""",1
"""11233""",1
"""20133""",1
…,…
"""00103""",1
"""00000""",1
"""20201""",1


In [9]:
ip_opt_list=(ip_opt.unique("msg").select(pl.col("msg")))["msg"].to_list()

In [10]:
ip_opt_list

['20133',
 '30033',
 '00103',
 '20201',
 '11233',
 '00023',
 '00000',
 '00013',
 '01113',
 '33222',
 '10033']

ipだけぴったりのmsgだけ適用すると外部行動者数が増加した

In [11]:
ip_opt_mod = (df.lazy()#ipにピッタリなmsgを他のシナリオに適用した場合。
    .join(ip_opt.lazy(), left_on=pl.col("msg"), right_on=pl.col("msg"))
    .group_by("ip_right").agg(pl.col("mean_num_xact").mean()).sort("mean_num_xact",descending=True)
    .select(pl.col("ip_right").alias("ip"),pl.col("mean_num_xact").alias("他のシナリオにmsgを適用したxact"))).collect()
ip_opt_frame=ip_opt_mod.join(ip_opt,left_on="ip",right_on="ip").select(pl.col("ip"),pl.col("msg"),pl.col("他のシナリオにmsgを適用したxact")).sort("他のシナリオにmsgを適用したxact",descending = True)
ip_opt_frame

ip,msg,他のシナリオにmsgを適用したxact
u64,str,f64
4583,"""00013""",51.040371
2821,"""00013""",51.040371
775,"""00023""",49.301484
2872,"""00023""",49.301484
2606,"""00023""",49.301484
…,…,…
4357,"""20133""",33.030645
4236,"""11233""",26.700332
1349,"""20201""",22.12748


In [13]:
ip_opt_frame.select(pl.col("他のシナリオにmsgを適用したxact").mean().alias("他のシナリオにmsgを適用したxactの期待値"))

他のシナリオにmsgを適用したxactの期待値
f64
38.987545


In [14]:
eval_df

key,value
str,f64
"""ev_by_opt""",36.35044
"""mean_of_opt_values""",55.363691
"""ev_by_ram""",51.689922
"""ev_by_urs""",23.456324
"""ram ev / opt ev""",1.421989
"""urs ev / opt ev""",0.645283
"""ram ev / opt mean""",0.933643
"""urs ev / opt mean""",0.423677


In [16]:
###ram
lf_rob_details = (
    df.lazy()
    .group_by("msg")
    .agg(pl.col(MEAN_NUM_XACT).mean())
    .sort(by="mean_num_xact", descending=True)
    # 上位5行を取得
).collect()
lf_rob_details.filter(pl.col("mean_num_xact")>=pl.col("mean_num_xact").mean()*1.8)

msg,mean_num_xact
str,f64
"""00003""",51.689922
"""00013""",51.040371
"""00023""",49.301484
"""00033""",48.019805
"""10003""",47.442031
…,…
"""10033""",43.777363
"""01023""",43.419805
"""20003""",42.579609


In [25]:
msg_mean=lf_rob_details.select(pl.col("mean_num_xact").mean())
msg_mean

mean_num_xact
f64
23.456324


In [18]:
# read parquet file
import polars as pl
# user_analysis= pl.read_parquet("../../experiments/user_analysis.parquet")
user_analysis= pl.scan_parquet("/Users/shinomilab-mac-pro/seokyeong/diffusion_seokyeong/test/twitch_ES_user_analysis.parquet")
user_analysis


In [19]:
ram_msg = lf_rob_details.filter(pl.col("mean_num_xact")>=pl.col("mean_num_xact").mean()*1.8).select(pl.col("msg"))
ram_msg_list =ram_msg["msg"].to_list()

ram_msg_us_an = user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list)).collect()

# # reverse_msg_list = lf_rob_details.lazy().sort("mean_num_xact",descending=True).select(pl.col("msg"))[-10:].to_list()
# # reverse_msg_us_an = user_analysis.lazy().filter(pl.col("msg").is_in(reverse_msg_list)).collect()


In [20]:
ram_msg_list

['00003',
 '00013',
 '00023',
 '00033',
 '10003',
 '00103',
 '10013',
 '00022',
 '01003',
 '10023',
 '00113',
 '01013',
 '00123',
 '00012',
 '00032',
 '10033',
 '01023',
 '20003',
 '10103',
 '00002']

In [22]:
lf_opt.sort("max_mean_num_xact",descending=True).collect()

us_id,ip,argmax_msg,max_mean_num_xact
u64,u64,str,f64
26,775,"""00023""",827.19
26,3905,"""10033""",656.46
9,3543,"""10033""",549.9
22,3905,"""00023""",489.86
22,775,"""00003""",473.12
…,…,…,…
14,18,"""00000""",0.0
8,18,"""00000""",0.0
1,18,"""00000""",0.0


In [27]:
lf_opt_list=(lf_opt
             .sort("max_mean_num_xact",descending=True).filter(pl.col("max_mean_num_xact")>23.456324
*1.8).collect())

lf_opt_list.group_by("argmax_msg").len().sort("len",descending=True)["argmax_msg"].to_list()
# lf_opt.sort("max_mean_num_xact",descending=True).collect()[:25]

['00003',
 '00013',
 '00023',
 '10003',
 '00033',
 '01003',
 '10013',
 '10023',
 '00103',
 '00113',
 '30013',
 '11003',
 '10033',
 '20003',
 '01013',
 '30003',
 '00022',
 '30033',
 '20033',
 '00002',
 '02013',
 '01103',
 '20013',
 '00213',
 '11023']

In [28]:
ip_opt_list

['20133',
 '30033',
 '00103',
 '20201',
 '11233',
 '00023',
 '00000',
 '00013',
 '01113',
 '33222',
 '10033']

00003,00033,00013,00002,00012は共通
つまり、少なくとも3ラウンドまで内部行動の一番強のメッセージを連続に伝播して、徐々に外部行動強のメッセージを伝播するのが外部行動者数を最大化してくれる。

In [18]:
reverse_msg_list

['33230',
 '23331',
 '32330',
 '23320',
 '33333',
 '33332',
 '33331',
 '33320',
 '23330',
 '33330']

In [29]:
ratio_xact_share_per = (user_analysis.lazy()
.select((pl.col("num_xact_of_users").mean() / pl.col("num_share_of_users").mean()).alias("xact_share_ratio")).collect())
xact_per_mean = (user_analysis.lazy().select(pl.col("num_xact_of_users").mean().alias("mean_num_xact_of_users")).collect())
share_per_mean = (user_analysis.lazy().select(pl.col("num_share_of_users").mean().alias("mean_num_share_of_users")).collect())

In [30]:
concat = pl.concat([xact_per_mean,share_per_mean,ratio_xact_share_per],how="horizontal")
concat

mean_num_xact_of_users,mean_num_share_of_users,xact_share_ratio
f64,f64,f64
0.102015,0.140003,0.728661


In [31]:
ram_ratio = (user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list))
        .select((pl.col("num_xact_of_users").mean() / pl.col("num_share_of_users").mean()).alias("xact_share_ratio")))
ram_mean_xact_of_users=user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list)).select(pl.col("num_xact_of_users").mean().alias("mean_num_xact_of_users"))
ram_mean_share_of_users=user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list)).select(pl.col("num_share_of_users").mean().alias("mean_num_share_of_users"))
ram = (
    pl.concat([ram_mean_xact_of_users, ram_mean_share_of_users, ram_ratio], how="horizontal").collect())
ram


mean_num_xact_of_users,mean_num_share_of_users,xact_share_ratio
f64,f64,f64
0.135628,0.151626,0.894487


In [32]:
xact_share=(user_analysis.lazy()
.group_by(["ip","us_id","msg"]).agg(pl.col("num_xact_of_users").sum().alias("xact"),pl.col("num_share_of_users").sum().alias("share"))
.group_by("msg").agg(pl.col("xact").mean(),pl.col("share").mean()).sort("xact",descending = True)
).collect()


In [33]:
xact_share

msg,xact,share
str,f64,f64
"""00003""",55.135917,73.945958
"""00013""",54.443063,60.110917
"""00023""",52.58825,48.805896
"""00033""",51.221125,39.975604
"""10003""",50.604833,65.597437
…,…,…
"""33320""",11.004542,17.306292
"""23300""",10.957854,31.208688
"""33330""",10.947583,13.621438


In [35]:
xact_share_mean= xact_share.lazy().select(pl.col("xact").mean(),pl.col("share").mean())
xact_share_ratio=xact_share.lazy().select([(pl.col("xact").mean()/pl.col("share").mean()).alias("xact/share")])
combine1 = pl.concat([xact_share_mean,xact_share_ratio],how="horizontal").collect()
combine1

xact,share,xact/share
f64,f64,f64
25.035909,34.394014,0.727915


In [36]:
ram_xact_share= (user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list))
.group_by(["ip","us_id","msg"]).agg(pl.col("num_xact_of_users").sum().alias("xact"),pl.col("num_share_of_users").sum().alias("share"))
.group_by("msg").agg(pl.col("xact").mean(),pl.col("share").mean()).sort("xact",descending = True)
).collect()
ram_xact_share

msg,xact,share
str,f64,f64
"""00003""",55.135917,73.945958
"""00013""",54.443063,60.110917
"""00023""",52.58825,48.805896
"""00033""",51.221125,39.975604
"""10003""",50.604833,65.597437
…,…,…
"""10033""",46.695854,34.425937
"""01023""",46.314458,39.981604
"""20003""",45.41825,56.720583


In [37]:
ram_xact_share_mean= ram_xact_share.select(pl.col("xact").mean(),pl.col("share").mean())
ram_xact_share_ratio=ram_xact_share.select([(pl.col("xact").mean()/pl.col("share").mean()).alias("xact/share")])
combine = pl.concat([ram_xact_share_mean,ram_xact_share_ratio],how="horizontal")
combine

xact,share,xact/share
f64,f64,f64
48.71794,54.466532,0.894456


In [5]:
opt_ratio = (opt_ratio_lazy.lazy().group_by(["msg","us_id","ip"])
        .agg(pl.col("num_xact_of_users").mean().alias("mean_num_xact_of_users"),
             pl.col("num_share_of_users").mean().alias("mean_num_share_of_users"),
            (pl.col("num_xact_of_users").mean() / pl.col("num_share_of_users").mean()).alias("xact_share_ratio")
        )
        .collect())
opt_ratio

msg,us_id,ip,mean_num_xact_of_users,mean_num_share_of_users,xact_share_ratio
str,u64,u64,f64,f64,f64
"""30010""",13,79973,0.134906,0.148113,0.910828
"""30000""",5,69023,0.06342,0.051303,1.23619
"""30000""",4,79973,0.047638,0.054216,0.878664
"""30002""",13,12605,0.02523,0.023032,1.095413
"""32221""",3,33649,0.45,0.71,0.633803
…,…,…,…,…,…
"""21113""",8,68985,0.226,0.247333,0.913747
"""33000""",11,38598,0.388505,0.373066,1.041383
"""30103""",9,9199,0.057205,0.092773,0.61661
